# 課程08－多智能體設計模式


## 設定


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 為何選擇多智能體系統？

現實世界的任務如旅程規劃涉及多種不同的專業知識—物流、本地知識、預算等等。單一智能體試圖處理一切會很快變得難以應付。

多智能體系統透過<strong>專精化</strong>來解決這問題：每個智能體聚焦於一個專業領域，產出比通才更高質素的結果。它們亦提升了<strong>可擴展性</strong>—你可以加入新的智能體（例如航班專家、餐館評論家），而無需重寫現有工作流程。這些智能體透過結構化流水線相互組合，將上下文從一者傳遞到下一者。


## 建立專門代理人


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## 建立一個序列工作流程

`WorkflowBuilder` 讓你把代理連接成一個有向圖。這裡我們建立一個簡單的兩步管道：**TravelPlanner** 起草行程，然後 **TravelConcierge** 審核並加強它。


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 為工作流程新增更多代理人

多代理模式其中一個最大的優點就是擴充非常容易。以下我們新增一個 **BudgetReviewer** 代理人，用來檢查計劃是否符合旅客的預算，標記可能會超出預算的項目，並建議節省開支的替代方案。現在工作流程會依序執行三個代理人：

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 總結

在這節課你學會了如何：

1. <strong>創建專門的代理人</strong> — 每個都有專注的角色（規劃、禮賓、預算審查）。
2. **使用 `WorkflowBuilder` 和 `add_edge` 將代理人連接成順序工作流程**。
3. <strong>串流輸出</strong> 多代理人管線，追蹤哪個代理人在說話。
4. <strong>擴展工作流程</strong>，通過新增代理人到鏈中而不修改現有代理人。

多代理人設計模式保持每個代理人簡單，同時產生比任何單一代理人更豐富、更經過徹底審查的結果。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
